# جمع‌آوری و ساخت پنل داده (Google Colab)
**هشدار مهم:** سرورهای Colab خارج از ایران‌اند و ممکن است codal.ir و tsetmc.com آنها را مسدود کنند. سلول ۲ این را بررسی می‌کند. اگر مسدود بود، مراحل جمع‌آوری را روی رایانه خودتان در ایران اجرا کنید، پوشه `data_out` را زیپ کنید و در سلول ۴ آپلود کنید؛ فقط ساخت پنل در Colab اجرا می‌شود.

In [ ]:
!pip -q install requests pandas openpyxl beautifulsoup4 lxml
import os
from google.colab import drive, files
drive.mount('/content/drive')
WORK='/content/drive/MyDrive/rohi_data'; os.makedirs(WORK,exist_ok=True); os.chdir(WORK)
print(os.getcwd())

In [ ]:
# سلول ۱ب: دریافت کدها از GitHub (شاخه پروژه؛ اگر مخزن خصوصی است آپلود دستی کنید)
!git clone -q -b claude/wizardly-archimedes-b9gs1c https://github.com/sarab2nya-cyber/rohi2.git /content/rohi2
!cp /content/rohi2/data_collection/*.py {WORK}/

In [ ]:
# سلول ۲: آزمون دسترسی
import requests
for u in ['https://codal.ir','https://search.codal.ir','https://cdn.tsetmc.com']:
    try: print(u, requests.get(u,timeout=15,headers={'User-Agent':'Mozilla/5.0'}).status_code)
    except Exception as e: print(u,'ERR',type(e).__name__)

In [ ]:
# سلول ۳: جمع‌آوری (فقط اگر سلول ۲ موفق بود)؛ ابتدا با --limit کوچک آزمون کنید
!python codal_collector.py list --from 1393/01/01 --to 1402/12/29
!python codal_collector.py fetch --limit 20
!python codal_collector.py parse
!python tsetmc_collector.py symbols
!python tsetmc_collector.py

In [ ]:
# سلول ۴: جایگزین سلول ۳ — آپلود data_out.zip ساخته‌شده در ایران
up=files.upload()
!unzip -oq data_out.zip -d .

In [ ]:
# سلول ۵: غربالگری و ساخت پنل
!python build_panel.py
import pandas as pd
df=pd.read_csv('data_out/panel.csv'); print(df.shape, df.Symbol.nunique())
df.describe().T.head(40)

In [ ]:
# سلول ۶: درصد مقادیر گمشده
print(df.isna().mean().sort_values(ascending=False).head(25))

In [ ]:
# سلول ۷: دانلود پنل برای استتا
files.download('data_out/panel.dta')